# 02 — Train your own tiny LLM on SEC filings

This notebook runs **inside the GPU pod** (you opened it from the JupyterLab URL the launcher printed).

What we'll do, in order:
1. Wait for the pod's background install to finish (only on first run).
2. Pick which SEC filing sections to train on.
3. Train a tiny **custom BPE tokenizer** on SEC text (~30 s).
4. Pretrain a **tiny GPT** for ~5–7 minutes (depth 4, ~5 M params).
5. Look at sample completions.
6. Bundle the model so you can download it.

This is genuinely Karpathy's nanochat under the hood — same tokenizer, same model, same training loop. We just point it at SEC filings instead of CommonCrawl, and we shrink the model + iteration count so it fits in 10 minutes.

## Step 0 — Wait for setup to finish

When the pod first booted, the launcher kicked off a background script that:
- cloned `nanochat`
- downloaded the SEC parquets from the workshop's storage pod
- set up a virtualenv with PyTorch + nanochat
- staged the parquets into nanochat's expected layout

JupyterLab itself comes up before that's done. The cell below polls `/workspace/startup.log` until it sees `Setup complete`. Usually 2–4 minutes on the first run; instant on re-runs.

In [ ]:
import time, pathlib

LOG = pathlib.Path('/workspace/startup.log')
start = time.time()
while True:
    txt = LOG.read_text() if LOG.exists() else ''
    if 'Setup complete' in txt:
        print(f'Setup complete after {int(time.time()-start)}s.')
        break
    if time.time() - start > 900:
        raise TimeoutError('startup didnt finish in 15 min — see /workspace/startup.log')
    last = txt.splitlines()[-1] if txt else '(no log yet)'
    print(f'  ... still installing. last line: {last[:100]}')
    time.sleep(8)

## Step 1 — Pick your data scope

The workshop's storage pod has four parquet files, one per SEC filing section:

| Scope          | Section                                                 |
|----------------|---------------------------------------------------------|
| `business`     | Item 1 — Business                                       |
| `market_risk`  | Item 7A — Quantitative & Qualitative Disclosures        |
| `mda`          | Item 7  — Management's Discussion & Analysis            |
| `risk_factors` | Item 1A — Risk Factors (the "we may be unable to" page) |
| `all`          | All four concatenated                                   |

For a 10-minute run, all four scopes will work. **`risk_factors`** has the most stylistically-distinctive text, so it's a great choice if you want recognizable output from the tiny model. **`all`** is the most representative of SEC filings overall.

In [ ]:
SCOPE = 'all'   # change to: 'business' | 'market_risk' | 'mda' | 'risk_factors' | 'all'
assert SCOPE in {'business', 'market_risk', 'mda', 'risk_factors', 'all'}
print(f'Using SCOPE = {SCOPE}')

In [ ]:
# Re-stage the parquets into nanochat's data dir for the chosen scope.
# (The launcher already did this with --scope all on first boot. If you
# changed SCOPE above, this re-runs the staging step.)
import subprocess, os
env = os.environ.copy()
env['PATH'] = '/workspace/nanochat/.venv/bin:' + env.get('PATH', '')
subprocess.run(
    ['/workspace/nanochat/.venv/bin/python', '/workspace/zero-to-llm/pod/prep_sec_data.py', '--scope', SCOPE],
    check=True, env=env,
)

## Step 2 — Train a custom BPE tokenizer

A tokenizer chops text into integer "tokens." We'll train a small one (vocab size 4096) directly on SEC text, so it captures phrases like " RISK FACTORS" or " forward-looking" as single tokens.

This is identical to what nanochat's speedrun does, just with `--max-chars` and `--vocab-size` lowered so it finishes in ~30 seconds instead of 30 minutes.

In [ ]:
%%bash
source /workspace/nanochat/.venv/bin/activate
cd /workspace/nanochat
python -m scripts.tok_train \
    --max-chars 20000000 \
    --vocab-size 4096

## Step 3 — Pretrain a tiny GPT

This is the meat of the workshop. We train a 4-layer GPT (~5 M parameters) for 400 optimizer steps on the SEC corpus. On a single A100 80 GB, that's ~5–7 minutes.

**Hyperparams worth knowing:**
- `--depth 4` → 4 transformer layers. Bigger = smarter but slower.
- `--max-seq-len 1024` → context length per training example.
- `--device-batch-size 8` → how many examples per GPU step. A100 80 GB can handle more, but small models don't need it.
- `--total-batch-size 32768` → tokens per *gradient step* (gradient accumulation handles the gap).
- `--num-iterations 400` → how many gradient steps total.
- `--core-metric-every -1` → skip the slow CORE benchmark eval.
- `--run dummy` → don't log to Weights & Biases.

Once training starts, you'll see lines like `step 100/400 | loss=4.85 | tok/s=...` scrolling. Loss should drop from ~7-8 down to ~3-4 over the run.

In [ ]:
%%bash
source /workspace/nanochat/.venv/bin/activate
cd /workspace/nanochat
python -m scripts.base_train \
    --depth 4 \
    --max-seq-len 1024 \
    --device-batch-size 8 \
    --total-batch-size 32768 \
    --num-iterations 400 \
    --eval-tokens 4096 \
    --eval-every 100 \
    --sample-every 100 \
    --core-metric-every -1 \
    --run dummy

## Step 4 — Look at some sample completions

Now we load the just-saved checkpoint and feed it a few SEC-flavored prompts to see what it learned. Remember: this is a **base** model, not a chat model — it does autocomplete, not Q&A. Treat each prompt as the *start* of a passage that the model continues.

In [ ]:
import sys
sys.path.insert(0, '/workspace/zero-to-llm/pod')
from _inference import load_base_engine, complete

print('Loading model...')
engine, tokenizer, meta, device = load_base_engine()
print(f'  device={device}, vocab={tokenizer.get_vocab_size()}')

PROMPTS = [
    'ITEM 1A. RISK FACTORS\n\nThe following risks could materially affect our',
    'Our business is focused on',
    'We may be unable to',
    'Management\'s Discussion and Analysis of Financial Condition\n\nOverview:',
]

for p in PROMPTS:
    print('\n' + '-' * 70)
    print('Prompt:', repr(p))
    print('Continuation:')
    print(p, end='')
    complete(engine, tokenizer, p, max_tokens=120, temperature=0.8, top_k=50)

## Step 5 — Try your own prompts

Run the cell below with whatever prompt you want. Re-run it (or change the prompt) as many times as you like.

In [ ]:
MY_PROMPT = 'The Company was incorporated in'
TEMPERATURE = 0.8
TOP_K = 50
MAX_TOKENS = 200

print(MY_PROMPT, end='')
complete(engine, tokenizer, MY_PROMPT, max_tokens=MAX_TOKENS, temperature=TEMPERATURE, top_k=TOP_K)

## Step 6 — (Optional) Save the model

This bundles the model checkpoint + tokenizer into `/workspace/sec_llm.tar.gz`. Once it's written, look in the JupyterLab file browser on the left, navigate to `/workspace`, right-click `sec_llm.tar.gz`, and choose **Download**.

In [ ]:
%%bash
source /workspace/nanochat/.venv/bin/activate
python /workspace/zero-to-llm/pod/save_model.py

## Step 7 — Done

You've trained an LLM end-to-end. Things you can try next:

- **Open `pod/03_chat.ipynb`** for an interactive chat-style REPL with your model.
- **Re-run with `SCOPE = 'risk_factors'`** for the most distinctively-styled output. (You'll need to re-run Step 2 + Step 3.)
- **Crank `--depth` up to 6 or 8** to see how a bigger model does. Each step doubles training time roughly, so depth=8 will take ~25 min.

**When you're done with the pod, terminate it** to stop charges:
[https://www.runpod.io/console/pods](https://www.runpod.io/console/pods)